# cv_lib — usage examples

A self-contained tour of the everyday helpers. We synthesise a tiny YOLO
dataset in a temp directory (no external data or model required) and walk
through:

1. **`inspect_dataset`** — dataset health check (missing labels, corrupt
   images, invalid boxes, class balance).
2. **`show_batch`** — grid of images with their ground-truth boxes.
3. **`find_errors` / `render_errors`** — FP/FN tiles, here comparing GT
   against *pre-saved* prediction labels (so no model load is needed).

> All cells run top-to-bottom with only `cv_lib` and its deps installed.

In [ ]:
%matplotlib inline
import tempfile
from collections import Counter
from pathlib import Path

import cv2
import numpy as np

# --- synthesise a tiny YOLO dataset: images/train + labels/train ---
root = Path(tempfile.mkdtemp(prefix='cvlib_demo_'))
img_dir = root / 'images' / 'train'
lbl_dir = root / 'labels' / 'train'
img_dir.mkdir(parents=True)
lbl_dir.mkdir(parents=True)

CLASS_NAMES = ['box', 'blob']
rng = np.random.default_rng(0)
H = W = 256

def draw_object(img, cx, cy, w, h, cls):
    """Paint a filled rectangle (cls 0) or circle (cls 1) and return its YOLO box."""
    x1, y1 = int((cx - w / 2) * W), int((cy - h / 2) * H)
    x2, y2 = int((cx + w / 2) * W), int((cy + h / 2) * H)
    color = (60, 180, 75) if cls == 0 else (235, 130, 40)
    if cls == 0:
        cv2.rectangle(img, (x1, y1), (x2, y2), color, -1)
    else:
        cv2.circle(img, ((x1 + x2) // 2, (y1 + y2) // 2), min(x2 - x1, y2 - y1) // 2, color, -1)
    return f'{cls} {cx:.5f} {cy:.5f} {w:.5f} {h:.5f}'

# 6 clean images, each with 1-2 objects
for i in range(6):
    img = np.full((H, W, 3), 30, dtype=np.uint8)
    lines = []
    for _ in range(rng.integers(1, 3)):
        cls = int(rng.integers(0, 2))
        cx, cy = rng.uniform(0.3, 0.7, size=2)
        w, h = rng.uniform(0.15, 0.3, size=2)
        lines.append(draw_object(img, cx, cy, float(w), float(h), cls))
    cv2.imwrite(str(img_dir / f'frame_{i:02d}.jpg'), img)
    (lbl_dir / f'frame_{i:02d}.txt').write_text('\n'.join(lines) + '\n')

# inject a couple of issues so the health check has something to report:
cv2.imwrite(str(img_dir / 'frame_06.jpg'), np.full((H, W, 3), 30, np.uint8))  # missing label
(img_dir / 'frame_07.jpg').write_bytes(b'not an image')                       # corrupt
(lbl_dir / 'frame_07.txt').write_text('0 0.5 0.5 0.2 0.2\n')

print('dataset root:', root)
print('images:', len(list(img_dir.glob('*.jpg'))))

## 1. Inspect the dataset

`inspect_dataset` scans a split and reports corrupt images, missing/empty
labels, out-of-range class ids or boxes, and the class distribution. The
labels directory is inferred from the `images/` path.

In [ ]:
from cv_lib import inspect_dataset

report = inspect_dataset(img_dir, num_classes=len(CLASS_NAMES), class_names=CLASS_NAMES)
report.print()

## 2. Visualise a batch with ground-truth boxes

`show_batch` accepts image paths (or arrays / tensors) and per-image label
arrays shaped `(N, 5)` = `[class_id, cx, cy, w, h]`, and returns the grid as
a BGR `np.ndarray` (also displayed inline).

In [ ]:
from cv_lib import show_batch

paths = sorted(str(p) for p in img_dir.glob('frame_0[0-5].jpg'))
labels = []
for p in paths:
    txt = (lbl_dir / (Path(p).stem + '.txt')).read_text().split()
    arr = np.array(txt, dtype=float).reshape(-1, 5)
    labels.append(arr)

grid = show_batch(paths, labels=labels, class_names=CLASS_NAMES, cols=3, tile_size=(200, 200))
print('grid shape:', grid.shape)

## 3. Find and render errors (GT vs predictions)

`find_errors` compares ground truth against predictions and returns FP / FN
entries. You can pass a model (`model_path=`) for live inference, or — as
here — a directory of pre-saved YOLO prediction `.txt` files
(`pred_labels_dir=`). We fake predictions by dropping one true box (→ **FN**)
and adding one spurious box (→ **FP**).

In [ ]:
from cv_lib import find_errors, render_errors

pred_dir = root / 'preds' / 'train'
pred_dir.mkdir(parents=True)

for i in range(6):
    gt = (lbl_dir / f'frame_{i:02d}.txt').read_text().strip().splitlines()
    pred_lines = []
    for ln in gt[1:]:  # drop the first GT box on each frame -> FN
        cls, *coords = ln.split()
        pred_lines.append(f"{cls} {' '.join(coords)} 0.90")  # add a conf column
    if i % 2 == 0:  # spurious detection on even frames -> FP
        pred_lines.append('1 0.15 0.15 0.1 0.1 0.80')
    (pred_dir / f'frame_{i:02d}.txt').write_text('\n'.join(pred_lines) + '\n')

errors = find_errors(img_dir, pred_labels_dir=pred_dir, conf_threshold=0.25)
print('error counts:', Counter(e.error_type for e in errors))

tiles = render_errors(errors, class_names=CLASS_NAMES, cols=4, max_tiles=12, tile_size=(200, 200))
print('tiles shape:', tiles.shape)

---
FP tiles are outlined in red, FN in blue. Swap `pred_labels_dir=pred_dir` for
`model_path='best.pt'` to run the same error analysis against a real
Ultralytics model.

See also `cvlib distribution`, `cvlib split`, and `cvlib compare` for the CLI
equivalents of these workflows.